In [1]:
import os
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
from joblib import Parallel, delayed
from scipy.spatial.distance import pdist
from google.colab import files

warnings.filterwarnings("ignore")

# ==========================================
# 1. Configuration for Dimensional Collapse Simulation
# ==========================================
N_TRIALS = 100
# Combinations of Demographic Categories (D) and Fuzzy Likert Questions (M)
DIMENSIONS_D = [10, 50, 100]
QUESTIONS_M = np.arange(5, 105, 10)
N_SAMPLES = 5000  # Fixed operational sample size for metric evaluation
K_CHOICES = 5
EPSILON = 0.001

output_dir = "hypothesis4_combinatorial_collapse"
os.makedirs(output_dir, exist_ok=True)

# ==========================================
# 2. Simulation Function: Metric Collapse (Distance Contrast)
# ==========================================
def simulate_metric_collapse(D, M, N):
    """
    Simulates the collapse of Euclidean distance contrast (Curse of Dimensionality)
    in a joint space of D demographic categories and M Likert-scale questions.
    """
    # 1. Generate base probabilities for D segments
    betas = 10 ** np.random.uniform(-1, 1.5, D)
    X_choice = np.random.uniform(0.01, 0.2, (D, M, K_CHOICES))

    P_cond = np.zeros((D, M, K_CHOICES))
    for d in range(D):
        for m in range(M):
            exp_val = np.exp(betas[d] * X_choice[d, m, :])
            P_cond[d, m, :] = exp_val / np.sum(exp_val)

    # 2. Sample responses for a single segment (to test intra-cluster variance)
    target_d = 0
    Y = np.zeros((N, M))
    for m in range(M):
        Y[:, m] = np.random.choice(np.arange(1, K_CHOICES+1), size=N, p=P_cond[target_d, m, :])

    # 3. Calculate pairwise Euclidean distances
    # Due to RAM limits, take a random subset for distance calculation
    subset_size = min(1000, N)
    Y_subset = Y[np.random.choice(N, subset_size, replace=False)]
    distances = pdist(Y_subset, metric='euclidean')

    if len(distances) == 0:
        return np.nan

    d_mean = np.mean(distances)
    d_std = np.std(distances)

    # Distance Contrast = Variance of Distance / Mean Distance
    # Approaches 0 in high dimensions, rendering k-means meaningless
    contrast = d_std / d_mean if d_mean > 0 else 0

    # Calculate Theoretical Critical Sample Size N* for the full space
    # Approximated by the state space complexity bounded by Fisher Info decay
    # N* = O(D * M * K / (epsilon^2 * mean(beta^2)))
    mean_beta_sq = np.mean(betas ** 2)
    n_star_theoretical = (D * M * (K_CHOICES - 1)) / (EPSILON**2 * (mean_beta_sq + 1e-5))

    return [contrast, n_star_theoretical]

# ==========================================
# 3. Grid Search Execution
# ==========================================
print("Executing combinatorial dimensionality simulation...")
results_summary = []

for D in DIMENSIONS_D:
    print(f"Scanning Demographic Dimension D = {D}...")
    for M in QUESTIONS_M:
        results = Parallel(n_jobs=-1)(
            delayed(simulate_metric_collapse)(D, M, N_SAMPLES) for _ in range(N_TRIALS)
        )
        results = np.array(results)

        mean_contrast = np.mean(results[:, 0])
        mean_n_star = np.mean(results[:, 1])

        results_summary.append({
            "Dimension_D": D,
            "Questions_M": M,
            "Total_Variables": D * M,
            "Distance_Contrast": mean_contrast,
            "Theoretical_N_star": mean_n_star
        })

df_res = pd.DataFrame(results_summary)
df_res.to_csv(f"{output_dir}/combinatorial_collapse_metrics.csv", index=False)

# ==========================================
# 4. Visualization
# ==========================================
plt.style.use('seaborn-v0_8-whitegrid')

# Graph 1: Collapse of Distance Contrast (Why k-means fails)
plt.figure(figsize=(10, 6))
colors = {10: 'tab:blue', 50: 'tab:green', 100: 'tab:red'}

for D in DIMENSIONS_D:
    subset = df_res[df_res["Dimension_D"] == D]
    plt.plot(subset["Questions_M"], subset["Distance_Contrast"],
             marker='o', color=colors[D], linewidth=2, label=f'Demographics D={D}')

plt.axhline(y=0.1, color='gray', linestyle='--', label='Metric Collapse Threshold (Contrast < 0.1)')
plt.title("Collapse of Euclidean Distance Contrast by Questionnaire Dimensions (M)")
plt.xlabel("Number of Fuzzy Questions (M)")
plt.ylabel("Relative Distance Contrast ($\sigma_d / \mu_d$)")
plt.legend()
plt.tight_layout()
plt.savefig(f"{output_dir}/fig1_distance_contrast_collapse.png", dpi=300)
plt.close()

# Graph 2: Explosion of Critical Sample Size N*
plt.figure(figsize=(10, 6))
for D in DIMENSIONS_D:
    subset = df_res[df_res["Dimension_D"] == D]
    plt.plot(subset["Questions_M"], subset["Theoretical_N_star"],
             marker='s', color=colors[D], linewidth=2, label=f'Demographics D={D}')

plt.yscale('log')
plt.title("Explosion of Critical Sample Size $N^*$ in Joint Categorical-Fuzzy Spaces")
plt.xlabel("Number of Fuzzy Questions (M)")
plt.ylabel("Critical Sample Size $N^*$ (log scale)")
plt.legend()
plt.tight_layout()
plt.savefig(f"{output_dir}/fig2_critical_sample_size_explosion.png", dpi=300)
plt.close()

# ==========================================
# 5. ZIP and Download
# ==========================================
zip_filename = "hypothesis4_combinatorial_collapse.zip"
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files_list in os.walk(output_dir):
        for file in files_list:
            zipf.write(os.path.join(root, file), arcname=file)

print(f"\nSimulation complete. Results compressed to: {zip_filename}")
files.download(zip_filename)

<>:120: SyntaxWarning: invalid escape sequence '\s'
<>:120: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_948/2823633303.py:120: SyntaxWarning: invalid escape sequence '\s'
  plt.ylabel("Relative Distance Contrast ($\sigma_d / \mu_d$)")


Executing combinatorial dimensionality simulation...
Scanning Demographic Dimension D = 10...
Scanning Demographic Dimension D = 50...
Scanning Demographic Dimension D = 100...

Simulation complete. Results compressed to: hypothesis4_combinatorial_collapse.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>